# Agent Comparison

This notebook compares the current Hanabi baselines:

- `RandomAgent`
- `BasicHeuristicAgent`
- `ConservativeHeuristicAgent`

It is meant to serve as the first lightweight record of agent metrics in the repository.

If you renamed the package recently, restart the notebook kernel before running the imports below.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

try:
    import hanabi_ai  # noqa: F401
    print("Using installed hanabi_ai package.")
except ModuleNotFoundError:
    if str(SRC_PATH) not in sys.path:
        sys.path.insert(0, str(SRC_PATH))
    print("hanabi_ai not installed; using src/ fallback.")

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from hanabi_ai.agents.random import RandomAgent
from hanabi_ai.agents.heuristic.basic import BasicHeuristicAgent
from hanabi_ai.agents.heuristic.conservative import ConservativeHeuristicAgent
from hanabi_ai.training.self_play import evaluate_self_play


In [ ]:
# Edit these values when you want to record a new comparison run.
PLAYER_COUNT = 2
GAME_COUNT = 200
SEED_BASE = 0
AGENT_SEED_BASE = 1000

In [ ]:
basic_evaluation = evaluate_self_play(
    lambda player_id, game_index: BasicHeuristicAgent(),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)

conservative_evaluation = evaluate_self_play(
    lambda player_id, game_index: ConservativeHeuristicAgent(),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)

random_evaluation = evaluate_self_play(
    lambda player_id, game_index: RandomAgent(
        seed=AGENT_SEED_BASE + (game_index * PLAYER_COUNT) + player_id
    ),
    player_count=PLAYER_COUNT,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
)


In [ ]:
def evaluation_to_row(name, evaluation):
    return {
        "agent": name,
        "games": evaluation.game_count,
        "players": evaluation.player_count,
        "avg_score": round(evaluation.average_score, 3),
        "median_score": round(evaluation.median_score, 3),
        "min_score": evaluation.min_score,
        "max_score": evaluation.max_score,
        "avg_turns": round(evaluation.average_turn_count, 3),
        "avg_hints": round(evaluation.average_hint_tokens, 3),
        "avg_strikes": round(evaluation.average_strike_tokens, 3),
        "avg_completed_stacks": round(evaluation.average_completed_stacks, 3),
        "win_rate": round(evaluation.win_rate, 4),
        "loss_rate": round(evaluation.loss_rate, 4),
        "score>=10": round(evaluation.score_at_least_10_rate, 4),
        "score>=15": round(evaluation.score_at_least_15_rate, 4),
    }


rows = [
    evaluation_to_row("RandomAgent", random_evaluation),
    evaluation_to_row("BasicHeuristicAgent", basic_evaluation),
    evaluation_to_row("ConservativeHeuristicAgent", conservative_evaluation),
]

In [ ]:
rows[0]

In [ ]:
rows[1]

In [ ]:
rows[2]

In [ ]:
headers = list(rows[0].keys())
widths = {
    header: max(len(header), max(len(str(row[header])) for row in rows))
    for header in headers
}

header_line = " | ".join(header.ljust(widths[header]) for header in headers)
separator_line = "-+-".join("-" * widths[header] for header in headers)

print(header_line)
print(separator_line)
for row in rows:
    print(" | ".join(str(row[header]).ljust(widths[header]) for header in headers))

In [ ]:
for name, evaluation in [
    ("RandomAgent", random_evaluation),
    ("BasicHeuristicAgent", basic_evaluation),
    ("ConservativeHeuristicAgent", conservative_evaluation),
]:
    print(name)
    # score_distribution is a tuple of (score, count) pairs.
    print(evaluation.score_distribution)
    print()